# ThermoRG Phase S1 v2 — Cooling Theory Validation

**Goal**: Validate BatchNorm and Skip as "cooling mechanisms"

**Design**:
- 4 configs: None / LN / BN / Skip
- 2 D values: base_ch 32, 96
- 100 epochs per run
- **Checkpoint every 20 epochs** — safe to interrupt
- **Logging every 10 epochs with ETA**
- **CPU smoke test before GPU run**

**Expected time**: ~50 min on T4 GPU

In [ ]:
import os, sys, subprocess, time, json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn


## ▶️ Step 1: Clone Code from GitHub

In [ ]:
%cd /kaggle/working
!rm -rf ThermoRG-NN
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret("gh_token")
repo_url = f"https://{gh_token}@github.com/xliu203/ThermoRG-NN.git"
!git clone {repo_url} --branch develop
%cd /kaggle/working/ThermoRG-NN
!git config --global user.email "xliu203@asu.edu"
!git config --global user.name "Leo Liu"
print("Cloned develop branch")

## ▶️ Step 2: Install Dependencies

In [ ]:
!pip install -q torch torchvision scipy numpy

sys.path.insert(0, "/kaggle/working/ThermoRG-NN")
sys.path.insert(0, "/kaggle/working/ThermoRG-NN/experiments/phase_s1")

import phase_s1_kaggle as p1

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## ▶️ Step 3: Smoke Test — Forward + Backward All 4 Configs

In [ ]:
import torch.nn as nn
from thermorg.simulations.networks.thermonet import ThermoNet

x = torch.randn(4, 3, 32, 32)
y = torch.randint(0, 10, (4,))

CONFIGS = [
    {"name": "None_NoSkip",  "layer_norm": False, "batch_norm": False, "use_skip": False},
    {"name": "LN_NoSkip",    "layer_norm": True,  "batch_norm": False, "use_skip": False},
    {"name": "BN_NoSkip",    "layer_norm": False, "batch_norm": True,  "use_skip": False},
    {"name": "None_Skip",    "layer_norm": False, "batch_norm": False, "use_skip": True},
]

print("=== Smoke Test: Forward + Backward ===")
for cfg in CONFIGS:
    model = ThermoNet(
        in_channels=3, num_classes=10, base_channels=32, depth=3,
        layer_norm=cfg["layer_norm"], batch_norm=cfg["batch_norm"],
        use_skip=cfg["use_skip"], dropout_rate=0.0, activation='gelu'
    )
    out = model(x)
    loss = nn.functional.cross_entropy(out, y)
    loss.backward()
    ok = out.shape == (4, 10) and not torch.isnan(loss)
    print(f"  {cfg['name']}: {'✅' if ok else '❌'} (loss={loss.item():.4f})")
    del model, out, loss

print("\n✅ Smoke test passed!")

## ▶️ Step 4: Run Phase S1 v2 Training

**Warning**: ~50 min on T4. Checkpoint resume enabled — safe to interrupt.

In [ ]:
# Config
CONFIGS = [
    ('None_NoSkip',  'none',       False),
    ('LN_NoSkip',    'layernorm',  False),
    ('BN_NoSkip',    'batchnorm',  False),
    ('None_Skip',    'none',       True),
]

D_VALUES = [32, 96]  # base_ch
SEEDS = [42]
EPOCHS = 100
BATCH_SIZE = 128
LR = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
CHECKPOINT_EVERY = 20
LOG_EVERY = 10

OUT_DIR = Path("/kaggle/working/phase_s1_results_v2")
OUT_DIR.mkdir(exist_ok=True)
CKPT_DIR = OUT_DIR / "checkpoints"
CKPT_DIR.mkdir(exist_ok=True)

# Get dataloaders
train_dl, val_dl, val_ds = p1.get_dataloaders(batch_size=BATCH_SIZE)
print(f"Train batches: {len(train_dl)}, Val samples: {len(val_ds)}")

# Training function
def train_one_model(model, epochs, lr, seed, checkpoint_path=None):
    start_epoch = 0
    best_loss = float('inf')
    
    if checkpoint_path and checkpoint_path.exists():
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        start_epoch = ckpt['epoch'] + 1
        best_loss = ckpt.get('best_loss', float('inf'))
        print(f"  ↪️  Resuming epoch {start_epoch}, best={best_loss:.4f}")
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    model = model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    for _ in range(start_epoch):
        scheduler.step()
    
    t0 = time.time()
    
    for epoch in range(start_epoch, epochs):
        model.train()
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
        scheduler.step()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X, y in val_dl:
                X, y = X.to(device), y.to(device)
                val_loss += criterion(model(X), y).item() * X.size(0)
        val_loss /= len(val_ds)
        if val_loss < best_loss:
            best_loss = val_loss
        
        if (epoch + 1) % LOG_EVERY == 0 or epoch == epochs - 1:
            elapsed = (time.time() - t0) / 60
            epochs_done = epoch - start_epoch + 1
            epochs_left = epochs - start_epoch - epochs_done
            eta = elapsed / epochs_done * epochs_left if epochs_done > 0 else 0
            print(f"  Epoch {epoch+1}/{epochs}: loss={val_loss:.4f}, best={best_loss:.4f}, elapsed={elapsed:.1f}min, ETA={eta:.1f}min")
        
        if checkpoint_path and (epoch + 1) % CHECKPOINT_EVERY == 0:
            torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                        'optimizer_state': optimizer.state_dict(), 'best_loss': best_loss}, checkpoint_path)
            print(f"  💾 Checkpoint at epoch {epoch+1}")
    
    return best_loss

# Main loop
print("\n" + "="*70)
print("STARTING Phase S1 v2 Training")
print("="*70)

results = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'epochs': EPOCHS,
    'total_runs': len(CONFIGS) * len(D_VALUES) * len(SEEDS),
    'configs': []
}

t_start = time.time()

for cfg_name, norm_type, use_skip in CONFIGS:
    print(f"\n{'='*60}")
    print(f"CONFIG: [{cfg_name}] norm={norm_type}, skip={use_skip}")
    print(f"{'='*60}")
    
    cfg_result = {'name': cfg_name, 'norm': norm_type, 'skip': use_skip}
    cfg_start = time.time()
    
    for base_ch in D_VALUES:
        for seed in SEEDS:
            run_name = f"{cfg_name}_ch{base_ch}_s{seed}"
            ckpt_path = CKPT_DIR / f"{run_name}.pt"
            
            print(f"\n--- {run_name} ---")
            
            model = ThermoNet(
                in_channels=3, num_classes=10, base_channels=base_ch, depth=3,
                layer_norm=(norm_type == 'layernorm'),
                batch_norm=(norm_type == 'batchnorm'),
                use_skip=use_skip, dropout_rate=0.0, activation='gelu'
            )
            
            t0 = time.time()
            best_loss = train_one_model(model, EPOCHS, LR, seed, ckpt_path)
            elapsed = (time.time() - t0) / 60
            
            print(f"  ✅ {run_name} DONE: best_loss={best_loss:.4f}, time={elapsed:.1f}min")
            
            key = f'ch{base_ch}'
            if key not in cfg_result:
                cfg_result[key] = {}
            cfg_result[key][f's{seed}'] = {'best_val_loss': float(best_loss), 'time_min': float(elapsed)}
            
            del model
            torch.cuda.empty_cache()
    
    cfg_time = (time.time() - cfg_start) / 60
    cfg_result['total_time_min'] = float(cfg_time)
    results['configs'].append(cfg_result)
    print(f"\n[{cfg_name}] total: {cfg_time:.1f} min")

total_time = (time.time() - t_start) / 60
results['total_time_min'] = float(total_time)
print(f"\n{'='*70}")
print(f"ALL DONE! Total time: {total_time:.1f} min")
print(f"{'='*70}")

# Save results
with open(OUT_DIR / 'phase_s1_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n=== SUMMARY ===")
for cfg in results['configs']:
    losses = [cfg[f'ch{ch}'][f's{s}']['best_val_loss'] for ch in D_VALUES for s in SEEDS if f'ch{ch}' in cfg]
    avg = sum(losses)/len(losses) if losses else 0
    print(f"{cfg['name']:<15} avg_loss={avg:.4f}  time={cfg.get('total_time_min', 0):.1f}min")

print("\n✅ Phase S1 v2 complete!")

## ▶️ Step 5: Push Results to GitHub

In [ ]:
OUT_DIR = "/kaggle/working/phase_s1_results_v2"
CKPT_DIR = f"{OUT_DIR}/checkpoints"
cmds = [
    f"git add {OUT_DIR}/*.json 2>/dev/null || true",
    f"git add {CKPT_DIR}/*.pt 2>/dev/null || true",
    f"git add experiments/phase_s1/notebooks/*.ipynb 2>/dev/null || true",
    'git commit -m "Data: Phase S1 v2 cooling theory validation from Kaggle" || true',
    'git push origin develop 2>&1 || true'
]

for cmd in cmds:
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                         cwd="/kaggle/working/ThermoRG-NN")
    if res.stdout.strip():
        print(res.stdout.strip())

print("\n✅ Results pushed to GitHub!")